# Practice 3 · Decoding and Evaluation

| Marker | Meaning |
|---|---|
| 🟢 EDIT | you write this part |
| 🟡 READ | read it, no need to change |
| ⚪ RUN | just run it as it is |

### I. Background

A language model produces a distribution over the next token. Decoding turns those
distributions into text, and an evaluation metric compares the resulting translations with
references. This practice uses fixed toy probabilities so that every decoding decision is
reproducible and fast to test.

### II. Task Requirements

1. **Q1** — write *greedy_decode(logits, eos_id)*.
2. **Q2** — write *beam_search(logits, beam_width, eos_id)* and run the supplied Top-p demo.
3. **Q3** — write *corpus_bleu(references, hypotheses, max_n=4)*.

Steps: open the notebook in Haima, write the 🟢 parts, and run the cells from top to bottom.

### III. Input and Output

| name | meaning | expected |
|---|---|---|
| *greedy_decode(...)* | best token at every step | token IDs without EOS |
| *beam_search(...)* | highest-scoring completed beam | token IDs without EOS |
| Top-p demo | stochastic sampling with fixed seed | reproducible token IDs |
| *corpus_bleu(...)* | corpus-level BLEU | a float in *[0, 1]* |
| references / hypotheses | tokenized translations | equal numbers of sentences |

### IV. Who does what

- **Q1**: required for undergraduate students; postgraduates do it as well.
- **Q2–Q3**: required for postgraduate students; optional for undergraduates.
  An undergraduate who finishes advanced questions gets bonus marks; leaving them unfinished costs nothing.


## Question 1 · Greedy decoding

Write **4–6 lines** in *greedy_decode(logits, eos_id)*. This function selects the most likely
token at every step and stops when the model produces EOS. The input has shape *(T, V)*, with
one row per decoding step. The same logits rows are used for every prefix in this toy experiment.

$$
y_t=\operatorname{argmax}_v\,x_{t,v}
$$

Return a Python list of token IDs **without EOS**. Do not sample or apply softmax: argmax is
unchanged whether you use logits or probabilities.

**You may find these functions useful:**

- *torch.argmax(step_logits)*: find the ID with the largest score.
- *.item()* and *int(...)*: convert a scalar tensor to a Python integer.
- *result.append(token_id)* and *break*: build the output and stop at EOS.


In [1]:
# 🟢 EDIT: set SID to the last two digits of your student ID.
SID = 23

import math
import random
from collections import Counter

import torch
import torch.nn.functional as F

torch.manual_seed(0)
random.seed(0)

# 🟡 READ: a fixed vocabulary and six rows of toy next-token scores.
VOCAB = ["<pad>", "<bos>", "<eos>", "I", "like", "this", "course",
         "We", "enjoy", "deep", "learning"]
EOS_ID = VOCAB.index("<eos>")
TOY_LOGITS = torch.full((6, len(VOCAB)), -5.0)
choices = [
    {3: 4.0, 7: 3.0},
    {4: 4.0, 8: 3.5},
    {5: 4.0, 9: 3.0},
    {6: 4.0, 10: 3.0},
    {2: 4.0, 6: 1.0},
    {2: 4.0, 5: 1.0},
]
for row, scores in zip(TOY_LOGITS, choices):
    for token_id, score in scores.items():
        row[token_id] = score


def greedy_decode(logits, eos_id):
    """Decode greedily and stop before EOS."""
    # 🟢 CODE HERE: write 4–6 lines.
    #   1) initialize an empty result list
    #   2) select argmax for each row
    #   3) break at EOS; otherwise append the token ID
    #   4) return the list
    # YOUR CODE HERE
    result = []
    for step_logits in logits:
        token_id = int(torch.argmax(step_logits).item())
        if token_id == eos_id: break
        result.append(token_id)
    return result
    


# ⚪ RUN: convert IDs back to text.
def ids_to_text(ids):
    return " ".join(VOCAB[i] for i in ids)


greedy_ids = greedy_decode(TOY_LOGITS, EOS_ID)
print("greedy IDs:", greedy_ids)
print("greedy text:", ids_to_text(greedy_ids))


greedy IDs: [3, 4, 5, 6]
greedy text: I like this course


## Question 2 · Beam search and stochastic decoding

Complete the core expansion in *beam_search(logits, beam_width, eos_id)* using the same
*(T, V)* toy logits as Q1. Initialization and result post-processing are already provided.
Write **8–12 lines** to expand active beams and retain the best candidates. Finished beams stop
at EOS. With *beam_width=1*, the result must match greedy decoding.

$$
\operatorname{score}(\text{prefix}+[v])=
\operatorname{score}(\text{prefix})+\log p(v\mid\text{step})
$$

Expand each active beam with its best *beam_width* next tokens, add log probabilities, and
keep the best *beam_width* candidates after each step.

**You may find these functions useful:**

- *torch.topk(log_probs, k=min(beam_width, step_logits.size(-1)))*.
- *zip(values.tolist(), indices.tolist())*: iterate over token scores and IDs.
- *candidates.append((prefix + [index], score + float(value)))*.
- *sorted(candidates, key=lambda item: item[1], reverse=True)[:beam_width]*.

The supplied Top-p function is not part of the implementation task. Run it with a fixed seed
and compare its output with greedy decoding.


In [2]:
def beam_search(logits, beam_width, eos_id):
    """Return the best sequence found by fixed-step beam search."""
    # 🟡 READ: initialization and result post-processing are provided.
    beams = [([], 0.0)]
    for step_logits in logits:
        log_probs = F.log_softmax(step_logits, dim=-1)
        candidates = []

        # 🟢 CODE HERE: write 8–12 lines for beam expansion and pruning.
        #   1) keep finished beams unchanged
        #   2) expand every active beam with top-k next tokens
        #   3) accumulate log probabilities
        #   4) retain the best beam_width candidates
        # YOUR CODE HERE
        for prefix, score in beams:
            if prefix and prefix[-1] == eos_id:
                candidates.append((prefix, score))
                continue
            values, indices = torch.topk(log_probs, k=min(beam_width, step_logits.size(-1)))
            for value, index in zip(values.tolist(), indices.tolist()):
                candidates.append((prefix + [index], score + float(value)))
        beams = sorted(candidates, key=lambda item: item[1], reverse=True)[:beam_width]

    # 🟡 READ: select the best sequence and remove its final EOS if present.
    best = max(beams, key=lambda item: item[1])[0]
    if best and best[-1] == eos_id:
        best = best[:-1]
    return best


# ⚪ RUN: beam search on the same fixed scores.
beam_ids = beam_search(TOY_LOGITS, beam_width=2, eos_id=EOS_ID)
print("beam IDs:", beam_ids)
print("beam text:", ids_to_text(beam_ids))


# 🟡 READ: supplied Top-p sampling; no implementation is required here.
def top_p_sample(step_logits, p=0.9, temperature=1.0, generator=None):
    scaled = step_logits / temperature
    probs = torch.softmax(scaled, dim=-1)
    sorted_probs, sorted_ids = torch.sort(probs, descending=True)
    cumulative = torch.cumsum(sorted_probs, dim=-1)
    keep = cumulative - sorted_probs < p
    keep[0] = True
    filtered = sorted_probs * keep
    filtered = filtered / filtered.sum()
    chosen = torch.multinomial(filtered, 1, generator=generator)
    return int(sorted_ids[chosen].item())


generator = torch.Generator().manual_seed(2026)
sampled_ids = []
for step_logits in TOY_LOGITS:
    token_id = top_p_sample(step_logits, p=0.9, temperature=1.2, generator=generator)
    if token_id == EOS_ID:
        break
    sampled_ids.append(token_id)
print("Top-p IDs:", sampled_ids)
print("Top-p text:", ids_to_text(sampled_ids))


beam IDs: [3, 4, 5, 6]
beam text: I like this course
Top-p IDs: [3, 8, 9, 10]
Top-p text: I enjoy deep learning


## Question 3 · Corpus BLEU

Complete *corpus_bleu(references, hypotheses, max_n=4)* for tokenized sentences. Each
reference and hypothesis is a list of strings, with one reference per hypothesis. The n-gram
counter, length checks, and loop scaffolding are provided.

Write **10–14 lines** across the two edit regions to accumulate clipped matches and compute
the final score. Do not add smoothing.

$$
p_n=\frac{\text{clipped n-gram matches}}{\text{candidate n-grams}},
\qquad BP=\min\left(1,\exp\left(1-\frac{r}{c}\right)\right)
$$

$$
\operatorname{BLEU}=BP\cdot\exp\left(\frac{1}{N}\sum_{n=1}^{N}\log p_n\right)
$$

Here *r* is the total reference length and *c* is the total hypothesis length. Return 0.0 for
an empty hypothesis or when any required precision is zero.

**You may find these operations useful:** *hyp_counts & ref_counts* for clipped n-gram counts,
*sum(counts.values())*, *zip(clipped, totals)*, *math.log*, and *math.exp*.


In [3]:
# 🟡 READ: this helper counts each n-gram in a tokenized sentence.
def ngram_counts(tokens, n):
    return Counter(tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1))


def corpus_bleu(references, hypotheses, max_n=4):
    """Compute unsmoothed corpus BLEU for one reference per hypothesis."""
    # 🟡 READ: validation, counters, and loop scaffolding are provided.
    if len(references) != len(hypotheses):
        raise ValueError("references and hypotheses must have the same length")
    clipped = [0] * max_n
    totals = [0] * max_n
    ref_length = 0
    hyp_length = 0
    for reference, hypothesis in zip(references, hypotheses):
        ref_length += len(reference)
        hyp_length += len(hypothesis)
        for n in range(1, max_n + 1):
            ref_counts = ngram_counts(reference, n)
            hyp_counts = ngram_counts(hypothesis, n)

            # 🟢 CODE HERE: write 2 lines to accumulate clipped matches and candidate counts.
            clipped[n - 1] += sum((hyp_counts & ref_counts).values())
            totals[n - 1] += sum(hyp_counts.values())

    if hyp_length == 0:
        return 0.0

    # 🟢 CODE HERE: write 8–12 lines for precision, geometric mean, and brevity penalty.
    #   1) compute each precision; use 1.0 when its denominator is zero
    #   2) return 0.0 when a required precision is zero
    #   3) compute the geometric mean and brevity penalty
    #   4) return their product as a float
    precisions = []
    for matches, total in zip(clipped, totals):
        precision = matches / total if total > 0 else 1.0
        precisions.append(precision)
    if any(p == 0 for p in precisions):
        return 0.0
    log_precision = sum(math.log(p) for p in precisions) / max_n
    geometric_mean = math.exp(log_precision)
    brevity_penalty = min(1.0, math.exp(1 - ref_length / hyp_length))
    return float(brevity_penalty * geometric_mean)


# ⚪ RUN: a small parallel test set and two sets of translations.
references = [
    ["I", "like", "this", "course"],
    ["We", "enjoy", "deep", "learning"],
    ["I", "enjoy", "this", "course"],
]
exact = [list(sentence) for sentence in references]
changed = [
    ["I", "enjoy", "this", "course"],
    ["We", "like", "deep", "learning"],
    ["I", "enjoy", "this", "course"],
]
print("BLEU(exact) =", round(corpus_bleu(references, exact), 4))
print("BLEU(changed) =", round(corpus_bleu(references, changed), 4))
greedy_tokens = [VOCAB[i] for i in greedy_ids]
beam_tokens = [VOCAB[i] for i in beam_ids]
print("greedy BLEU on one sentence =", round(corpus_bleu(references[:1], [greedy_tokens]), 4))
print("beam BLEU on one sentence =", round(corpus_bleu(references[:1], [beam_tokens]), 4))


BLEU(exact) = 1.0
BLEU(changed) = 0.4762
greedy BLEU on one sentence = 1.0
beam BLEU on one sentence = 1.0
